In [8]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import sys
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
#  自动定位项目根目录（兼容 Jupyter 和命令行）
# ============================================================

def get_current_dir():
    """获取当前执行环境的工作目录"""
    try:
        # 如果是 Jupyter，使用 get_ipython 或 os.getcwd()
        from IPython import get_ipython
        if get_ipython() is not None:
            return Path(os.getcwd())
    except ImportError:
        pass
    # 命令行环境：尝试 __file__，若不存在则回退到当前工作目录
    try:
        return Path(__file__).parent.resolve()
    except NameError:
        return Path.cwd()

def find_project_root(start_path):
    """向上查找包含 'data/raw' 的目录作为项目根"""
    current = Path(start_path).resolve()
    for parent in [current] + list(current.parents):
        if (parent / "data" / "raw").exists():
            return parent
    # 如果找不到，尝试当前工作目录
    return Path.cwd()

# 获取脚本所在目录或当前工作目录
script_dir = get_current_dir()
project_root = find_project_root(script_dir)
data_root = project_root / "data"

RAW_DIR = data_root / "raw" / "KolektorSDD"
PROCESSED_DIR = data_root / "processed"
SPLITS_DIR = data_root / "splits"
TARGET_SIZE = (500, 500)

print("="*60)
print(f"当前工作目录: {Path.cwd()}")
print(f"推测的项目根目录: {project_root}")
print(f"原始数据目录: {RAW_DIR}")
print(f"处理后数据目录: {PROCESSED_DIR}")
print(f"数据集划分目录: {SPLITS_DIR}")
print("="*60)

# 检查原始数据是否存在
if not RAW_DIR.exists():
    raise SystemExit(f"错误：未找到原始数据目录 {RAW_DIR}\n"
                     f"请确保数据解压后位于 data/raw/KolektorSDD 下，"
                     f"或者手动设置 project_root 变量。")

# ============================================================
#  步骤1：递归收集所有图像和对应的掩码
# ============================================================

image_paths = []
mask_paths = []

# 支持多种图像扩展名（不区分大小写）
image_exts = [".jpg", ".jpeg", ".JPG", ".JPEG", ".bmp", ".png"]
# 可能的掩码命名规则（按优先级尝试）
mask_candidates = ["_label.bmp", ".bmp", "_mask.bmp", "_labels.bmp"]

print("正在扫描图像文件...")
for ext in image_exts:
    for img_path in RAW_DIR.rglob(f"*{ext}"):
        # 避免重复处理
        if img_path in image_paths:
            continue
        stem = img_path.stem
        found = False
        for candidate in mask_candidates:
            mask_path = img_path.parent / f"{stem}{candidate}"
            if mask_path.exists():
                image_paths.append(img_path)
                mask_paths.append(mask_path)
                found = True
                break
        if not found:
            print(f"⚠️  警告：{img_path.name} 未找到对应掩码，跳过")

print(f"✅ 共找到 {len(image_paths)} 对图像-掩码配对。")

if len(image_paths) == 0:
    raise SystemExit("错误：未找到任何有效配对，请检查文件命名规则。")

# ============================================================
#  步骤2：按 70% / 15% / 15% 划分数据集
# ============================================================

relative_paths = [str(p.relative_to(RAW_DIR)) for p in image_paths]

train_val, test = train_test_split(
    relative_paths, test_size=0.15, random_state=42, shuffle=True
)
train, val = train_test_split(
    train_val, test_size=0.15 / 0.85, random_state=42, shuffle=True
)

# 创建分割目录并保存索引文件
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
with open(SPLITS_DIR / "train.txt", "w") as f:
    f.write("\n".join(train))
with open(SPLITS_DIR / "val.txt", "w") as f:
    f.write("\n".join(val))
with open(SPLITS_DIR / "test.txt", "w") as f:
    f.write("\n".join(test))

print(f"训练集: {len(train)}, 验证集: {len(val)}, 测试集: {len(test)}")

# ============================================================
#  步骤3：统一尺寸，处理图像和掩码，提取边界框并生成标注CSV
# ============================================================

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / "images").mkdir(exist_ok=True)
(PROCESSED_DIR / "masks").mkdir(exist_ok=True)

annotations = []  # 存储所有标注信息

for i, rel_path in enumerate(relative_paths):
    img_path = RAW_DIR / rel_path
    stem = img_path.stem
    # 找到对应的掩码（利用之前配对，直接从 mask_paths 获取）
    mask_path = None
    for mp in mask_paths:
        if mp.parent == img_path.parent and mp.stem.startswith(stem):
            mask_path = mp
            break
    if mask_path is None:
        print(f"⚠️  跳过 {img_path.name}：无法定位掩码")
        continue

    # 读取图像
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"⚠️  无法读取图像：{img_path}")
        continue
    h_orig, w_orig = img.shape

    # 读取掩码
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"⚠️  无法读取掩码：{mask_path}")
        continue
    if mask.shape[:2] != img.shape:
        print(f"⚠️  掩码尺寸({mask.shape})与图像({img.shape})不匹配，将resize掩码")
        mask = cv2.resize(mask, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)

    # Resize 图像和掩码
    resized_img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
    # 二值化掩码（确保只有0和255）
    resized_mask = (resized_mask > 0).astype(np.uint8) * 255

    # 保存图像和掩码（命名：用下划线拼接相对路径避免重名）
    safe_name = rel_path.replace(os.sep, "_").replace("..", "")
    # 去除原有扩展名
    base_name = safe_name.rsplit('.', 1)[0] if '.' in safe_name else safe_name
    img_save_path = PROCESSED_DIR / "images" / f"{base_name}.jpg"
    mask_save_path = PROCESSED_DIR / "masks" / f"{base_name}_mask.png"
    cv2.imwrite(str(img_save_path), resized_img)
    cv2.imwrite(str(mask_save_path), resized_mask)

    # 提取边界框
    has_defect = np.any(resized_mask > 0)
    if has_defect:
        contours, _ = cv2.findContours(resized_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            # 归一化坐标（YOLO格式）
            x_center = (x + w / 2) / TARGET_SIZE[0]
            y_center = (y + h / 2) / TARGET_SIZE[1]
            width = w / TARGET_SIZE[0]
            height = h / TARGET_SIZE[1]
            annotations.append({
                "image": f"{base_name}.jpg",
                "class": 0,
                "x_center": x_center,
                "y_center": y_center,
                "width": width,
                "height": height,
                "has_defect": 1,
                "orig_path": rel_path
            })
    else:
        annotations.append({
            "image": f"{base_name}.jpg",
            "class": -1,
            "x_center": -1,
            "y_center": -1,
            "width": -1,
            "height": -1,
            "has_defect": 0,
            "orig_path": rel_path
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(relative_paths):
        print(f"处理进度: {i+1}/{len(relative_paths)}")

# ============================================================
#  保存标注CSV
# ============================================================

df = pd.DataFrame(annotations)
df.to_csv(PROCESSED_DIR / "annotations.csv", index=False)

print("\n" + "="*60)
print("✅ 预处理完成！")
print(f"总图像数: {len(relative_paths)}")
print(f"含缺陷图像数: {df[df['has_defect']==1]['image'].nunique()}")
print(f"边界框总数: {len(df[df['has_defect']==1])}")
print(f"无缺陷图像数: {df[df['has_defect']==0]['image'].nunique()}")
print("="*60)
print("生成的文件：")
print(f"  - 图像: {PROCESSED_DIR / 'images'}")
print(f"  - 掩码: {PROCESSED_DIR / 'masks'}")
print(f"  - 标注: {PROCESSED_DIR / 'annotations.csv'}")
print(f"  - 划分索引: {SPLITS_DIR}")

当前工作目录: c:\Users\19840\Desktop\data\scripts
推测的项目根目录: C:\Users\19840\Desktop
原始数据目录: C:\Users\19840\Desktop\data\raw\KolektorSDD
处理后数据目录: C:\Users\19840\Desktop\data\processed
数据集划分目录: C:\Users\19840\Desktop\data\splits
正在扫描图像文件...
✅ 共找到 798 对图像-掩码配对。
训练集: 558, 验证集: 120, 测试集: 120
处理进度: 20/798
处理进度: 40/798
处理进度: 60/798
处理进度: 80/798
处理进度: 100/798
处理进度: 120/798
处理进度: 140/798
处理进度: 160/798
处理进度: 180/798
处理进度: 200/798
处理进度: 220/798
处理进度: 240/798
处理进度: 260/798
处理进度: 280/798
处理进度: 300/798
处理进度: 320/798
处理进度: 340/798
处理进度: 360/798
处理进度: 380/798
处理进度: 400/798
处理进度: 420/798
处理进度: 440/798
处理进度: 460/798
处理进度: 480/798
处理进度: 500/798
处理进度: 520/798
处理进度: 540/798
处理进度: 560/798
处理进度: 580/798
处理进度: 600/798
处理进度: 620/798
处理进度: 640/798
处理进度: 660/798
处理进度: 680/798
处理进度: 700/798
处理进度: 720/798
处理进度: 740/798
处理进度: 760/798
处理进度: 780/798
处理进度: 798/798

✅ 预处理完成！
总图像数: 798
含缺陷图像数: 104
边界框总数: 178
无缺陷图像数: 694
生成的文件：
  - 图像: C:\Users\19840\Desktop\data\processed\images
  - 掩码: C:\Users\19840\Desktop\data\processed\ma